# EUR to INR Transfer Outlook
### A Data-Driven Approach to Remittance Timing

**Project documentation notebook**  
**Institution:** Otto-von-Guericke University Magdeburg  
**Project team:** Abhay Kishore · Jayakrishna Ajithkumar Jayasree · Rosa Mariam John

---
# 1. Project Overview

## 1.1 What is our project?

**EUR to INR Transfer Outlook** is a data science project focused on the timing of money transfers from Germany to India. It investigates whether historical exchange-rate data and machine learning can help people decide **when is the best time to transfer euros (EUR) into Indian rupees (INR)**.

The amount of INR received from a transfer depends on the EUR–INR exchange rate at the time of conversion. Because the rate changes over time, sending the same number of euros on different days can produce different amounts of INR.

Our proposed solution is a **weekly outlook dashboard** that helps users compare transferring today with waiting during the next five working days. Rather than promising the best day, the dashboard is intended to communicate the model's estimated probabilities and uncertainty.

## 1.2 Why does this problem matter?

Many international students and workers in Germany send money to India regularly. A common approach is to transfer on a fixed day of the month, such as payday, without evaluating the current exchange rate.

This raises a practical question: **could a data-informed choice of transfer day lead to more INR than following a fixed monthly schedule?**

For example, if someone transfers **€500**, a rate of **₹100 per €1** would imply **₹50,000**, while **₹101 per €1** would imply **₹50,500** giving a difference of **₹500**, before any transfer fees or provider-specific exchange-rate margins. This is why timing may matter.

## 1.3 Problem Statement

The EUR–INR exchange rate fluctuates, but people making recurring transfers generally lack a simple, data-driven way to assess whether today's rate compares favorably with rates that may occur over the next week.

The project therefore explores two related issues:

1. **Predictive usefulness:** Can historical information provide a meaningful short-term outlook for EUR–INR rate movements?
2. **Practical usefulness:** Would following that outlook improve the INR received compared with transferring on a *fixed day every month*?

An important part of the investigation is identifying conditions in which the model is uncertain or may not outperform a simple baseline.

## 1.4 Research Questions

Our proposal defines four research questions:

1. **Short-term prediction:** Can a machine learning model predict whether the EUR–INR rate will be higher than today's rate on each of the next five working days, better than chance?
2. **Transfer performance:** Does an outlook-based transfer strategy yield more INR than fixed-day monthly transfers?
3. **Relevant factors:** Which factors are associated with movements in the EUR–INR exchange rate?
4. **Market conditions:** Does model performance differ between calm and volatile market periods?

## 1.5 Project Objectives and Expected Deliverables

Our planned work aims to:

- Analyze the historical behavior of the EUR–INR exchange rate and its relationship with selected external factors.
- Prepare a time-aligned dataset from publicly available exchange-rate, crude-oil, and interest-rate decision information.
- Build machine learning models that estimate whether the EUR–INR rate will be higher than today's rate at horizons of **one to five working days**.
- Examine different market conditions, including calm and volatile periods.
- *Backtest an outlook-based monthly transfer strategy against a fixed-day baseline and the best possible day in each month.*
- Present findings through an interactive dashboard showing the outlook, uncertainty, recent trends, and historical performance.

These are **planned deliverables**, not claims that the models or dashboard have already been built.

---
# 2. Proposed Data and Methodology

## 2.1 Planned Data Sources

| Source | Planned information | Purpose |
|---|---|---|
| **Frankfurter API (ECB reference rates)** | Daily EUR–INR and EUR–USD exchange rates | Main exchange-rate series; derive USD–INR |
| **FRED API (Federal Reserve)** | Daily crude-oil prices | Explore relationships with currency movement |
| **ECB and RBI websites** | Dates of interest-rate decisions | Build features related to monetary-policy events |



## 2.2 Planned Workflow

We intend to follow the steps below, documenting the actual implementation and observations in later sections.

| Stage | What we plan to do | Why it matters |
|---|---|---|
| **1. Data collection** | Retrieve exchange rates, oil prices, and policy decision dates | Assemble the information relevant to the research questions |
| **2. Data cleaning and alignment** | Check missing days, duplicates, and outliers; align series using ECB working days | Make the different sources comparable |
| **3. Exploratory data analysis** | Explore trends, volatility, calendar effects, and relationships | Understand data behavior and guide feature design |
| **4. Feature and target engineering** | Compute historical indicators and five forward-looking binary targets | Provide model inputs and measurable prediction outcomes |
| **5. Time-based splitting and modeling** | Separate training, validation, and test periods; train and compare models | Evaluate predictions on later, unseen data |
| **6. Market-condition analysis** | Cluster or group periods by market behavior | Investigate whether performance changes across conditions |
| **7. Backtesting** | Replay monthly transfers using fixed-day and outlook-based rules | Assess practical value in INR, not just classification accuracy |
| **8. Dashboard and interpretation** | Present forecasts, uncertainty, and historical evidence | Make findings understandable to prospective users |

## 2.3 How We Plan to Evaluate the Project

We will evaluate the project from two perspectives:

**Prediction quality**

- Accuracy, F1 score, and ROC-AUC for each of the five forecast horizons.
- Probability calibration: whether predicted likelihoods match observed frequencies.
- Performance across calm and volatile periods.

**Transfer-strategy performance**

- Relative improvement in average INR received per euro over a fixed-day monthly baseline.
- Proportion of months in which the outlook-based approach outperforms the baseline.
- Share of hindsight-perfect timing gains captured.
- Statistical testing of whether observed improvements are distinguishable from random variation.

The planned backtest sends the **same euro amount once per month** under each strategy. The outlook-based strategy chooses the first day that meets its calibrated decision rule, or the final working day if no such signal occurs.

---
# 3. Data Collection

## 3.1 Why did we collect data from different sources?

The EUR–INR exchange rate is the central variable in our project. However, exchange-rate movements can be associated with several other factors, including international currency markets, commodity prices, monetary policy, global risk, economic activity, and major news events. To investigate these possible relationships, we collected datasets covering multiple aspects of the economy rather than relying only on historical EUR–INR rates.

We divided the collection work among **three team members**, with each person responsible for five categories. The datasets were then brought together into a shared **`raw/` collection folder**. This folder serves as the starting point for our subsequent data inspection, cleaning, and integration steps.

## 3.2 Division of Responsibilities

| Team member | Main area | Categories assigned |
|---|---|---|
| **Abhay** | Financial markets and macroeconomic prices | Currency legs; energy; metals and other commodities; interest rates; inflation |
| **Rosa** | Capital flows and economic fundamentals | Capital flows; risk sentiment; trade and external balance; reserves and intervention; growth indicators |
| **Jayakrishna** | News, events and remittance behaviour | Uncertainty and geopolitics; event calendar; text and sentiment; seasonality; remittances |

Each member focused on a different group of potential influences. This made the data collection more manageable and helped us cover financial, economic, and event-driven aspects of the problem.

## 3.3 Abhay — Financial Markets and Macroeconomic Prices

I collected datasets representing currency movements, globally traded commodities, monetary policy, and inflation. These indicators help us examine whether the EUR–INR rate changes alongside other market prices and economic conditions.

### A. Currency legs

**Collected files and sources:**
- **European Central Bank (ECB):** `ECB_EUR_INR_Exchange_Rate_Daily_2020_2026.csv`, `ECB_EUR_USD_Exchange_Rate_Daily_2020_2026.csv`, `ECB_USD_INR_Exchange_Rate_Daily_2020_2026.csv`.
- **FRED:** `FRED_USD_INR_Exchange_Rate_Daily_2020_2026.csv`, `FRED_US_Broad_Dollar_Index.csv`.
- **Reserve Bank of India (RBI):** `RBI_INR_Exchange_Rates_Daily.xlsx`.

**Why we collected them:** EUR–INR is our target exchange rate. EUR–USD and USD–INR provide related currency legs that can help us examine whether movements in the euro, US dollar, or rupee coincide with changes in EUR–INR. The broad US dollar index provides an additional measure of overall dollar strength. RBI quotes provide an alternative exchange-rate source for comparison.

**How they may be used:** To plot historical currency trends, verify consistency between related currency pairs and data providers, calculate daily returns, and construct lagged or rolling exchange-rate indicators. We must avoid accidentally using future-day rates in predictive features.

### B. Energy

**Collected files and sources:**
- **FRED:** `FRED_Global_Brent_Crude_Oil_Price_Daily_2020_2026.csv`, `FRED_US_WTI_Crude_Oil_Price_Daily_2020_2026.csv`, `FRED_US_Henry_Hub_Natural_Gas_Price_Daily_2020_2026.csv`.

**Why we collected them:** Energy prices can influence import costs, inflation, and market expectations, potentially affecting exchange rates—particularly for economies exposed to energy imports.

**How they may be used:** To analyze whether major energy-price changes precede or accompany EUR–INR movements, using changes and lagged values instead of assuming a direct causal relationship.

### C. Metals and other commodities

**Collected file and source:**
- **World Bank:** `WorldBank_Global_Commodity_Prices_Monthly.xlsx`.
- **Reserve Bank of India (RBI):**`RBI_India_Monetary_Policy_Rates_And_Reserve_Requirements.xlsx`

**Why we collected it:** Metals and other commodity prices provide additional information about global demand, supply conditions, and inflationary pressures.

**How it may be used:** To compare commodity-price trends with exchange-rate behavior. Because this workbook is monthly, it will require careful calendar alignment and publication-date checks before being combined with daily data.

### D. Interest rates

**Collected files and sources:**
- **FRED:** `FRED_EuroArea_ECB_Deposit_Facility_Rate_2020_2026.csv`, `fred_ECBDFR.csv`, `fred_DFEDTARU.csv`, `FRED_US_10Y_Treasury_Yield_Daily_2020_2026.csv`.
- **Reserve Bank of India (RBI):** `RBI_India_Forex_Reserves_Weekly.xlsx`, `RBI_India_Central_Bank_Balance_Sheet.xlsx`, `RBI_India_Reserve_Money_Components.xlsx`.

**Why we collected them:** Interest rates and monetary-policy differences may influence currency demand, capital movements, and exchange-rate expectations.

**How they may be used:** To examine interest-rate levels and changes, compare policy conditions across economies, and construct indicators around monetary-policy shifts. Some files contain overlapping ECB rate series; they are alternative representations, not independent observations.

### E. Inflation

**Collected files and sources:**
- **Eurostat:** `Eurostat_EuroArea_HICP_Inflation_Monthly_2020_2026.xlsx`, `Eurostat_EuroArea_HICP_Inflation_Annual_2020_2026.xlsx`.
- **India's Ministry of Statistics and Programme Implementation (MOSPI):** `MOSPI_India_CPI_2012_Base_Current_2026.xlsx`, `MOSPI_India_CPI_2012_Base_Historical_Through_2026.xlsx`, `MOSPI_India_CPI_2024_Base_Current_2026.xlsx`, `MOSPI_India_CPI_2024_Base_Historical_Through_2026.xlsx`, `cpi_base2012.csv`, `cpi_base2024.csv`.
- **Reserve Bank of India (RBI):** `RBI_India_Central_Bank_Balance_Sheet.xlsx`, `RBI_India_Reserve_Money_Components.xlsx`,`RBI_India_Monetary_Policy_Rates.csv`.
- **CBOE:** `CBOE_US_VIX_Volatility_Index.csv`.


**Why we collected them:** Inflation can be connected to interest-rate decisions, purchasing power, and longer-term currency conditions.

**How they may be used:** To compare euro-area and Indian inflation trends and consider inflation-related features. India CPI files include **different base years (2012 and 2024)**; these series cannot be combined naively as though their index values used the same reference base.

## 3.4 Rosa — Capital Flows and Economic Fundamentals

Rosa collected datasets representing capital flows, financial-market risk sentiment, trade and external balances, foreign-exchange reserves, and economic growth. These indicators help us examine whether EUR–INR exchange-rate movements are associated with changes in international financial conditions and broader economic fundamentals.

### A. Capital flows

**Collected files and sources:**
- **FRED:** `BAA10Y.csv` and `BAMLH0A0HYM2.csv`.

**Why we collected them:** Capital movements and financing conditions can influence currency demand and exchange-rate pressure. Central-bank balance-sheet data, reserve-money components, and market credit spreads provide information about monetary and financial conditions. However, these datasets are **indirect financial-condition indicators**, rather than direct measures of foreign portfolio investment or other cross-border capital flows.

**How they may be used:** To analyze changes in financial conditions alongside EUR–INR movements and explore whether credit spreads and monetary indicators provide useful explanatory information. Direct capital-flow datasets may be required for a more complete analysis.

### B. Risk sentiment

**Collected files and sources:**
- **FRED:** `BAA10Y.csv`, `BAMLH0A0HYM2.csv`, and `VIXCLS.csv`

**Why we collected them:** Global financial uncertainty and changes in investor risk preferences can influence capital movements and currency markets. The VIX provides information about expected US equity-market volatility, while credit spreads and the broad US dollar index provide additional indicators of financial-market conditions.

**How they may be used:** To compare EUR–INR movements during periods of relatively high and low market uncertainty, examine relationships between risk indicators and exchange-rate changes, and evaluate whether predictive models behave differently across market conditions. `VIXCLS.csv` and `CBOE_US_VIX_Volatility_Index.csv` appear to contain overlapping volatility data and should be checked before use to avoid duplication.

### C. Trade and external balance

**Collected files and sources:**
- **ECB:** `trade_monthly_raw.csv`, which contains a `source` column referring to ECB balance-of-payments series.
- **FRED:** `XTEXVA01INM667S.csv`, `XTIMVA01INM667S.csv`, containing Indian trade indicators.
- **Manually prepared dataset:** `india_cab_manual.csv`, which contains a current-account-balance-related header but no data rows.

**Why we collected them:** International trade and current-account balances provide information about cross-border payments and demand for foreign currencies. Changes in exports, imports, and external balances may be associated with longer-term exchange-rate movements.

**How they may be used:** To examine relationships between trade conditions, external balances, and EUR–INR trends. Because trade and balance-of-payments indicators are generally reported at lower frequencies than daily exchange rates, careful calendar alignment and publication-date handling will be necessary. The empty `india_cab_manual.csv` file cannot be used until valid observations are added and verified.

### D. Reserves and intervention

**Collected files and sources:**
- **Reserve Bank of India (RBI):** `RBI_India_Forex_Reserves_Weekly.xlsx`, `reserves.xlsx`, `RBI_India_Central_Bank_Balance_Sheet.xlsx`, `RBI_India_Reserve_Money_Components.xlsx`.

**Why we collected them:** Foreign-exchange reserves and central-bank monetary statistics provide information about India's external financial position and monetary conditions. Reserve holdings can also offer context for understanding the central bank's capacity to intervene in currency markets. However, **changes in foreign-exchange reserves are not direct measures of currency intervention**, because reserves can also change due to valuation effects and other factors.

**How they may be used:** To study reserve trends alongside EUR–INR exchange-rate movements and explore whether changes in reserve holdings are associated with currency-market conditions. The weekly reserve data will require alignment with daily exchange rates. `RBI_India_Forex_Reserves_Weekly.xlsx` and `reserves.xlsx` may contain overlapping observations and should be compared before analysis.

### E. Growth indicators

**Collected files and sources:**
- **Eurostat:** `growth_raw.csv`, which includes a `source` column identifying Eurostat indicators.
- **India's Ministry of Statistics and Programme Implementation (MOSPI):** `gdp_growth_base2011-12.csv`, `gdp_growth_base2022-23.csv`, `iip_general.csv`, `iip_general_base2011-12.csv`,  `iip_general_base2022-23.csv`.

**Why we collected them:** Economic growth and industrial production provide information about overall economic performance. Differences in growth conditions between India and the euro area may influence investor expectations, capital movements, and longer-term exchange-rate behavior.

**How they may be used:** To compare economic growth and industrial activity with EUR–INR exchange-rate trends and investigate whether these indicators provide useful explanatory or predictive features. The Indian GDP and Index of Industrial Production (IIP) files contain **different base-year versions (2011–12 and 2022–23)**, which should not be combined without checking their definitions and methodological differences. Because growth indicators are reported less frequently than exchange rates, publication dates and reporting frequencies must also be considered when integrating them with daily data.

## 3.5 Jayakrishna — News, Events and Remittance Behaviour

IJayakrishna collected datasets representing geopolitical risk, economic-policy uncertainty, economic event calendars, and news sentiment. I also identified historical exchange-rate data that can be used to study seasonal patterns and simulate remittance timing. These indicators help us examine whether EUR–INR exchange-rate movements are associated with major events, changes in market uncertainty, news coverage, and recurring calendar patterns.

### A. Uncertainty and geopolitics

**Collected files and sources:**
- **Geopolitical Risk (GPR) Index:** `gpr_daily_2026-10-28.xls` and `gpr_monthly_2026-10-28.xls`.
- **FRED:** `fred_EUEPUINDXM.csv`, `fred_INDEPUINDXM.csv`, and `fred_USEPUINDXD.csv`, containing economic-policy-uncertainty indicators for Europe, India, and the United States.

**Why we collected them:** Geopolitical developments and economic-policy uncertainty can influence investor confidence, financial-market conditions, and exchange-rate volatility. These indicators provide additional information about global and regional uncertainty that may not be fully reflected in currency prices alone.

**How they may be used:** To examine whether periods of heightened geopolitical risk or economic-policy uncertainty coincide with larger EUR–INR movements and compare model performance during relatively stable and uncertain periods. The datasets contain different reporting frequencies, including daily and monthly observations, so careful date alignment and publication-lag handling will be necessary. The original source of the GPR workbooks should also be verified before analysis.

### B. Event calendar

**Collected files and sources:**
- **FRED:** `fred_release_dates_10.csv`, `fred_release_dates_50.csv`.

**Why we collected them:** Scheduled economic announcements and monetary-policy decisions can influence market expectations and may be associated with sudden exchange-rate movements. Event dates provide a way to examine whether EUR–INR behaves differently around important economic releases.

**How they may be used:** To construct event-date indicators, analyze exchange-rate volatility around scheduled announcements, and investigate whether economic releases are associated with short-term currency movements. The FRED files contain release identifiers and dates, but **the exact meaning of release IDs 10 and 50 must be verified** before assigning event labels. Announcement timing and information availability must also be considered to avoid introducing future information into predictive features.

### C. Text and sentiment

**Collected files and sources:**
- **GDELT:** `gdelt_rupee_count.csv`, `gdelt_rupee_tone.csv`.

**Why we collected them:** News coverage can provide information about public attention, economic developments, and market sentiment. Changes in the volume and overall tone of news mentioning the Indian rupee may be associated with exchange-rate uncertainty or shifts in market expectations.

**How they may be used:** To analyze whether news volume and sentiment indicators coincide with EUR–INR movements and explore their usefulness as explanatory or predictive features. The datasets contain `datetime` and `value` fields, which may allow them to be aligned with exchange-rate observations. However, the original GDELT queries, the interpretation of the tone values, and the availability of news aggregates at prediction time must be verified before use.

### D. Seasonality

**Relevant files and sources:**
- **European Central Bank (ECB):** `ECB_EUR_INR_Exchange_Rate_Daily_2020_2026.csv`.

**Why we included this category:** Exchange-rate movements and remittance activity may exhibit recurring patterns around weekdays, month-end periods, salary payment cycles, holidays, or particular months of the year. Studying these patterns can help determine whether calendar timing is associated with differences in EUR–INR behavior.

**How they may be used:** To construct weekday, month, quarter, and month-end indicators from historical dates and test whether statistically meaningful seasonal patterns exist. These features may also help evaluate different monthly remittance-timing strategies. **No separate seasonality dataset was identified** in the collected archive, so the required calendar indicators will be derived from existing historical observations.

### E. Remittances

**Relevant data and sources:**
- **European Central Bank (ECB):** `ECB_EUR_INR_Exchange_Rate_Daily_2020_2026.csv`, which provides historical EUR–INR rates for simulating transfer timing.

**Why we included this category:** Remittance timing is the main practical application of the project. Changes in the EUR–INR exchange rate affect the indicative amount of Indian rupees received when transferring a fixed amount of euros. Examining historical exchange-rate patterns can help assess whether choosing different transfer dates may improve the value of recurring remittances.

**How they may be used:** To simulate equal-size monthly EUR-to-INR transfers using historical exchange rates and compare alternative timing strategies against a fixed-date or other baseline approach. The analysis can measure differences in the indicative INR amounts across simulated strategies. However, **these simulations do not represent actual individual remittance behavior**, and real transfer outcomes may differ because of provider-specific fees, exchange-rate spreads, and processing times. These additional costs should only be included if reliable provider data becomes available.

## 3.6 What is inside the shared `raw/` folder?

All the datasets collected are kept as original source material so we can inspect and process them systematically in the next stage.

The files cover several frequencies:
- **Daily:** exchange rates, market prices, volatility, geopolitical risk, and some news indicators.
- **Weekly:** Indian foreign-exchange reserves.
- **Monthly or quarterly:** inflation, trade, economic growth, industrial production, and several uncertainty measures.
- **Event-based:** dated release schedules and monetary-policy observations.

This mixed structure is important: we **cannot simply join every file on the same date**. Each source has its own observation calendar, release schedule, and possible revisions. During preprocessing we will examine the date fields, sampling frequency, missing values, duplicates, and publication timing before deciding what should be included in the final modeling dataset.

### Initial collection-stage observations

1. **Repeated or overlapping indicators:** ECB deposit rates, VIX, forex reserves, Indian CPI, GDP, and IIP have multiple files or versions. They should be compared and standardized before selecting a canonical series.
2. **An empty file:** `india_cab_manual.csv` contains only the columns `quarter`, `cab_usd_bn`, and `release_date` and has no observations.
3. **Different base years:** Indian CPI and growth-related datasets include alternative base-year series. We must avoid joining index levels from different bases without appropriate handling.
4. **Different data frequencies and release times:** Even when an observation is dated for a particular month, the statistic might have been published later. Features must use information available by the simulated decision date.
5. **Not every category has a standalone dataset:** Seasonality can be engineered from dates; direct remittance-behavior records have not been identified in this archive.

**Result of this step:** We assembled a broad raw-data collection and documented which indicators are available, their origins, and their intended analytical roles.

---
# 4. Initial Data Cleaning and Calendar Alignment

## 4.1 Purpose of this stage

The raw datasets were collected from different sources in CSV and Excel formats. They have different column structures, date formats, and reporting frequencies (daily, weekly, monthly, and event-based). Before combining the data, we used separate Python scripts in `src/` to clean each source and prepare comparable date columns.

**The goal is to prepare and check the data** The original files remain unchanged in `data/raw/`. The resulting CSV files are saved in `data/interim/`, while some data-quality reports stored in `reports/quality/`.

## 4.2 General structure of the cleaning scripts

The scripts follow a similar sequence:

1. **Read the source:** Load an original CSV or Excel file from `data/raw/`.
2. **Check the structure:** Confirm expected columns, date fields, and source-specific information.
3. **Standardize values:** Convert dates and numeric values to consistent formats; normalize names and units where needed.
4. **Identify data-quality issues:** Check missing values, invalid entries, duplicate observations, and unusual changes.
5. **Prepare the time dimension:** Sort by date and expand weekly, monthly, or event-based data to a daily calendar.
6. **Save outputs:** Write the cleaned CSV to `data/interim/` and quality reports to `reports/quality/`.

The scripts are **source-specific**: not every dataset requires the same cleaning or frequency conversion. 

## 4.3 Main data-quality checks

| Check | Reason |
|---|---|
| Expected columns and date field | Ensures the source can be interpreted correctly |
| Date conversion and date range | Detects invalid dates and identifies available history |
| Missing values | Reveals incomplete observations |
| Duplicate records or dates | Avoids accidental repeated observations |
| Numeric conversion and value validity | Detects non-numeric or invalid measurements |
| Unusual changes and date gaps | Highlights observations requiring review |
| Reporting frequency and calendar alignment | Prepares datasets for later comparison and merging |

**Important:** Flags are prompts for review; a missing value or large change does not automatically mean the underlying source is wrong.

## 4.4 Inspect the interim files

Run these cells from the project root.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
QUALITY_DIR = PROJECT_ROOT / "reports" / "quality"
SRC_DIR = PROJECT_ROOT / "src"

if not INTERIM_DIR.exists() and (PROJECT_ROOT / "interim").exists():
    INTERIM_DIR = PROJECT_ROOT / "interim"
if not QUALITY_DIR.exists() and (PROJECT_ROOT / "quality").exists():
    QUALITY_DIR = PROJECT_ROOT / "quality"

print("Interim directory:", INTERIM_DIR)
print("Quality directory:", QUALITY_DIR)

To list the CSV files created during preprocessing.

In [ ]:
interim_files = sorted(INTERIM_DIR.glob("*.csv"))
quality_files = sorted(QUALITY_DIR.glob("*.csv"))

print("Interim CSV files:", len(interim_files))
print("Quality report CSV files:", len(quality_files))

pd.DataFrame({"interim_file": [path.name for path in interim_files]})

### Example: Inspect the cleaned EUR–INR exchange-rate dataset

We use the cleaned ECB EUR–INR dataset as an example, rather than loading every interim file at once. Some combined datasets are large and contain many columns.

In [ ]:
eur_inr_path = INTERIM_DIR / "ECB_EUR_INR_Exchange_Rate_Daily_2020_2026_Clean.csv"

if eur_inr_path.exists():
    eur_inr = pd.read_csv(eur_inr_path, parse_dates=["date"])
    display(eur_inr.head())
    print("Shape:", eur_inr.shape)
    print("Date range:", eur_inr["date"].min().date(), "to", eur_inr["date"].max().date())
else:
    print("File not found:", eur_inr_path)

To check missing values, duplicate dates, and data types.

In [ ]:
if eur_inr_path.exists():
    print("Missing values:")
    print(eur_inr.isna().sum())
    print("\nDuplicate dates:", eur_inr["date"].duplicated().sum())
    print("\nData types:")
    print(eur_inr.dtypes)

## 4.5 Calendar alignment

The source datasets do not all publish observations on the same dates. Some are daily, while others are weekly, monthly, or tied to events. Separate preprocessing scripts therefore create daily-format versions of selected datasets. Where appropriate, earlier observed values may be carried forward until another observation is available.

The interim folder includes both `*_Clean.csv` files and some `*_Daily.csv` or `*_Clean_Daily.csv` files. These are different stages of preprocessing and should not be treated as separate independent sources.

**Limitation:** A daily row does not necessarily mean a new value was observed on that day. Also, calendar expansion does not by itself guarantee that a value was publicly known on that date. We will verify publication timing and use the intended ECB working-day calendar before modeling.

In [ ]:
# Show examples of datasets with daily-aligned versions.
daily_files = [path.name for path in interim_files
               if path.name.endswith("_Daily.csv") or path.name.endswith("_Clean_Daily.csv")

print("Files with a daily-format suffix:", len(daily_files))
pd.DataFrame({"daily_aligned_file": daily_files}).head(20)

## 4.6 Validate the processed CSV files with `check_processed.py`

The script `check_processed.py` is an **additional validation tool**, rather than a cleaning script. It reads CSV files and prints a short report showing the date range, estimated frequency, invalid or duplicate dates, unusually large date gaps, missing-value percentages, non-numeric values, and suspicious jumps.

It returns **`READY`** when none of its blocking rules is triggered and **`CHECK`** when the file needs further inspection. In the current script, missingness above **20%** and unusual jumps are highlighted for review; missingness above **50%**, non-numeric values, date problems, or excessive date gaps can trigger `CHECK`.

In [ ]:
import subprocess
import sys

check_script = SRC_DIR / "check_processed.py"
if not check_script.exists():
    check_script = PROJECT_ROOT / "scripts" / "src" / "check_processed.py"

if check_script.exists() and eur_inr_path.exists():
    result = subprocess.run(
        [sys.executable, str(check_script), str(eur_inr_path)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("Could not locate the validation script or the example CSV.")

To check **all** interim CSV files using the same script, run the following command from the project root. This may take longer because some interim datasets are large:

```bash
python src/check_processed.py data/interim
```

## 4.7 Summary of this preprocessing stage

- We used individual Python scripts to clean source data from `data/raw/` without overwriting the originals.
- The main checks covered structure, dates, missing values, duplicates, numeric validity, and unusual observations.
- Selected weekly, monthly, and event-based datasets were converted to daily-format CSV files for easier integration.
- The resulting datasets were stored in `data/interim/`, with available quality reports in `reports/quality/`.
- `check_processed.py` provides an additional readiness check, but its `READY` result does not mean that every modeling requirement is satisfied.

---
# 5. Dataset Selection and Integration

## 5.1 We combined dataset?

After the initial cleaning and calendar alignment, the datasets were available as separate CSV files in `data/interim/`. For the research questions, we need to examine the EUR–INR exchange rate together with possible explanatory factors on a consistent timeline.

We therefore used the separate script `merge_data.py` to select relevant datasets and combine them into **`data/processed/data_daily.csv`**.

**Goal:** One row per calendar day with the EUR–INR rate (when published), related indicators and information about important calendar and policy events.

## 5.2 Which datasets did we select, and why?

We collected more data than we eventually needed for the first combined dataset. The merge script uses selected cleaned files and already-consolidated topic tables, rather than merging every raw file. Selection was based on (1) relevance to EUR–INR movements and our research questions, (2) availability over the study period, (3) the possibility of consistent date alignment, and (4) avoiding unnecessary overlap or repeated versions of the same information.

| Selected data group | Main variables or files used | Why it may be relevant |
|---|---|---|
| **EUR–INR exchange rate (target)** | ECB daily EUR–INR; `eur_inr`, `eur_inr_prev` | The outcome we want to understand and later predict; yesterday's rate is a historical reference. |
| **Related currency rates** | ECB EUR–USD and USD–INR | Helps describe movements of the euro and Indian rupee relative to the US dollar. |
| **Investment flows** | `fpi_clean_daily`: equity/debt/total foreign portfolio flows, 1- and 4-week measures | International capital flows may be associated with demand for Indian assets and the rupee. |
| **Economic growth and inflation** | `growth_clean_daily`: Euro Area and India inflation, GDP, unemployment, economic sentiment, industrial production, gaps and changes | Represents different macroeconomic conditions in the two economies. |
| **Financial-market risk** | `risk_clean_daily`: VIX, BAA spread, changes and rolling averages | Provides indicators of global financial stress and risk appetite. |
| **Trade and current accounts** | `trade_clean_daily`: India exports/imports/balance, Euro Area current account and goods balance | External trade and payment flows may influence currency demand. |
| **India foreign-exchange reserves** | `reserves_daily`: total reserves and components, changes | Provides context on the external position and reserve movements. |
| **Energy and US financial conditions** | FRED Brent oil, US 10-year Treasury yield, broad US-dollar index | Oil costs and global dollar/interest-rate conditions may be associated with EUR–INR changes. |
| **Central-bank policy** | RBI repo rate, ECB deposit rate, calculated India–Euro Area rate gap | Captures policy settings and interest-rate differences between the two economies. |
| **Geopolitical and policy uncertainty** | `gpr_clean`, `us_epu_clean`, `gpr_india_clean`, `in_epu_clean`, `eu_epu_clean` | Provides measures of uncertainty that may coincide with currency-market changes. |
| **Calendar and scheduled events** | `calendar_clean`, `events_clean`: weekdays, holidays, payday window, Diwali, monetary-policy decisions, US CPI/jobs and India budget | Enables investigation of calendar patterns and whether known events relate to rate movements. |

### Why were other datasets not included?

The raw collection also contains overlapping datasets, alternative versions, and indicators that are not referenced by this merge script. We did **not** merge every file just because it was available. Including every dataset could introduce repeated information, incompatible reporting dates, unnecessary columns, and greater cleaning and validation complexity.

Some selected `interim/` files are themselves combined or daily-aligned datasets, so a single input CSV can contribute several features. **This is an initial, research-driven feature selection**, not evidence that the omitted variables are irrelevant. We may revisit the selection after EDA and model validation.

## 5.3 How does `merge_data.py` create `data_daily.csv`?

The script uses a repeated pattern:

1. **Read prepared files:** `read()` loads each selected CSV from `data/interim/` and parses its `date` column.
2. **Build a daily timeline:** Create one row for every calendar day starting on **1 January 2020**, then join the EUR–INR target to its actual observation dates.
3. **Identify ECB days:** Add `is_ecb_day` based on weekdays and the ECB/TARGET closing-day rules used by the script.
4. **Join explanatory datasets:** Add several daily columns directly and attach the most recently available information from other datasets using `pd.merge_asof()`.
5. **Account for availability:** `add()` constructs a `known_from` date, sometimes adding an availability lag, so source values are not automatically treated as known at the beginning of their observation date.
6. **Create additional features:** Calculate the RBI–ECB interest-rate gap, event flags, days until scheduled events, and counts/flows accumulated since the preceding ECB fixing.
7. **Save and check:** Write `data/processed/data_daily.csv`, print coverage and missingness information, check source duplicates, and verify that nonmissing EUR–INR values occur only on marked ECB days.

The resulting file is a **daily integration table**. It is not yet the final modeling matrix: target labels, temporal train/validation/test splits, and further feature validation still need to be created.

### Important: How the script handles dates and information availability

The central helper `add(base, df, lag_days=0)` removes completely empty source rows, defines when an observation is considered known, and attaches the latest known observation to each daily row through an as-of merge. This preserves the latest available lower-frequency value until a newer one becomes available.

The script deliberately applies different handling depending on the data source:

| Source | Treatment in the script | Reason |
|---|---|---|
| Historical EUR–INR and the related ECB exchange rates | Previous-day value (`lag_days=1`) | Avoid using today's ECB fixing as an explanatory variable for today's own rate. |
| Brent oil, US Treasury yield, broad USD index | Previous-day value (`lag_days=1`) | Their US-market observations may become available after the relevant European fixing time. |
| Daily geopolitical risk and US policy-uncertainty series | Previous-day value | A conservative availability assumption. |
| Monthly India/EU uncertainty series | Move to month-end and add 7 calendar days | Approximation for reporting delay after the measured month. |
| RBI/ECB policy-rate levels | Merge on their provided dates | The script treats rate levels as known from those dates. |
| Calendar flags and scheduled events | Attach by date; calculate time to upcoming events | Dates of scheduled events can be known ahead of the events themselves. |

## 5.4 Load the integrated dataset

In [ ]:
from pathlib import Path
import pandas as pd

# The notebook is expected to be in the project root.
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "data_daily.csv"

# Load the merged output.
if DATA_PATH.exists():
    data_daily = pd.read_csv(DATA_PATH, parse_dates=["date"])
    print("Dataset loaded successfully")
    display(data_daily.head())
else:
    print(f"File not found: {DATA_PATH}")
    print("Run src/merge_data.py first, or update PROJECT_ROOT.")

## 5.5 Inspect the final structure

**What:** Check the number of observations and variables, date coverage, data types, and duplicate dates.

**Why:** We need to verify that the integration produced a consistent daily table before examining relationships or building a prediction model.

In [ ]:
if DATA_PATH.exists():
    print("Rows and columns:", data_daily.shape)
    print("Start date:", data_daily["date"].min().date())
    print("End date:", data_daily["date"].max().date())
    print("Duplicate dates:", data_daily["date"].duplicated().sum())
    print("Number of numeric variables:", data_daily.select_dtypes(include="number").shape[1])
    print("Missing days in the daily calendar:",
          len(pd.date_range(data_daily["date"].min(),
                            data_daily["date"].max()).difference(data_daily["date"])))

## 5.6 Check target availability and missing values

**What:** Count how many dates contain an actual published EUR–INR value and summarize missing values in the other columns.

**Why:** The output includes weekends and non-ECB days, when the target may not be published. Those missing target values should not be filled as if they were actual new fixings.

In [ ]:
if DATA_PATH.exists():
    print("Total calendar days:", len(data_daily))
    print("Available EUR–INR observations:", data_daily["eur_inr"].notna().sum())
    print("Missing EUR–INR observations:", data_daily["eur_inr"].isna().sum())
    print("Marked ECB days:", data_daily["is_ecb_day"].sum())

    missing_summary = (data_daily.isna().sum()
                       .to_frame(name="missing_count"))
    missing_summary["missing_percent"] = (
        missing_summary["missing_count"] / len(data_daily) * 100
    ).round(2)
    display(missing_summary[missing_summary["missing_count"] > 0]
            .sort_values("missing_count", ascending=False).head(20))

## 5.7 Check that the target follows the ECB-day calendar

**What:** Look for observations on non-ECB days or target values missing from marked ECB days.

**Why:** The project's target is based on ECB reference-rate observations, not an independently recorded exchange rate for every calendar day. A missing target on a marked ECB day may signal a publication gap or a coverage issue and should be inspected.

In [ ]:
if DATA_PATH.exists():
    target_on_closed_day = data_daily[
        data_daily["eur_inr"].notna() & (data_daily["is_ecb_day"] == 0)
    ]
    missing_on_ecb_day = data_daily[
        data_daily["eur_inr"].isna() & (data_daily["is_ecb_day"] == 1)
    ]

    print("Rates on non-ECB days:", len(target_on_closed_day))
    print("Marked ECB days with missing rates:", len(missing_on_ecb_day))
    display(missing_on_ecb_day[["date", "eur_inr", "is_ecb_day"]].head(10))

## 5.8 Result and interpretation

For the uploaded version of `data_daily.csv`, the integration produced:

- **2,500 calendar-day records** covering **2020-01-01 to 2026-11-04**.
- **106 columns**: one date column, the EUR–INR target and a broad set of candidate explanatory features.
- **1,750 nonmissing EUR–INR observations** and **750 dates without a target**.
- **No duplicate dates** in the combined output.

This means the merging stage has created a consistent, date-indexed table suitable for subsequent inspection. It **does not yet show** that all indicators improve predictive performance or that all historical information was available in real time.

---
# 6. Final Cleaning and ECB Fixing-Day Feature Dataset

## 6.1 Why was another cleaning step needed?

The merged `data_daily.csv` contains **one row per calendar day**. This is useful while collecting information from different sources because economic releases, market observations, and events may happen on different days. However, the EUR–INR reference rate is **not published every calendar day**.

For modeling the exchange rate, we need one row for each day on which an EUR–INR fixing was actually available. The script `daily_report_final_clean.py` therefore converts the merged daily table into a more focused **ECB fixing-day feature dataset** called `features_ecb_clean.csv`.

This is a **final selection and validation step**, not another collection of raw datasets: the selected economic indicators are retained, but unnecessary rows and duplicated forms of some variables are removed.

## 6.2 General workflow of `daily_report_final_clean.py`

| Step | What the script does | Why it matters |
|---|---|---|
| Read merged data | Loads `data/processed/data_daily.csv` and parses `date` | Starts from the integrated dataset instead of the original raw files |
| Keep EUR–INR fixing days | Retains only rows where `eur_inr` is available | Removes weekends, holidays, and other dates without a published target |
| Apply start date | Keeps dates on or after **8 January 2020** | Removes the earliest observations used for initialization / warm-up |
| Remove redundant features | Drops raw event/flow columns already summarized by `_since_fix` columns, and any columns constant on fixing days | Avoids keeping duplicate or uninformative versions of the same information |
| Organize columns | Assigns every remaining variable to a named group | Makes the feature table easier to inspect and use |
| Standardize selected types | Converts calendar, event, and indicator columns to integers | Makes categorical flags and day counts consistent |
| Validate | Runs missingness, chronology, range, and event-conservation checks | Catches issues before EDA or feature engineering |
| Export | Saves `data/processed/features_ecb_clean.csv` and `docs/gap_log.csv` | Creates a reproducible modeling base and an audit trail |

**Important:** The script does **not** impute all missing data. It explicitly permits the first few missing values in three four-week foreign portfolio investment (FPI) variables, since these are caused by their required historical window.

## 6.3 Why keep the `_since_fix` variables?

A market or scheduled event can take place after the last ECB fixing or on a day when the EUR–INR rate is not published. If we simply remove non-fixing days, those daily events may disappear from the dataset.

The earlier merge script therefore creates variables such as `is_rbi_decision_day_since_fix` and `fpi_total_day_since_fix`. These summarize events or daily flows **since the previous fixing**. The final-cleaning script retains these summaries and removes their corresponding plain daily columns.

For example, when an event happens over a weekend, its occurrence can still be represented in the feature row for the next available fixing day. This preserves useful information while changing from a calendar-day table to a fixing-day table.

## 6.4 How are the final columns organized?

The code groups all retained columns into the following categories. These groups describe possible explanatory factors rather than results proving that any factor predicts EUR–INR.

| Group | Example features | Reason to retain |
|---|---|---|
| **Target** | `eur_inr`, `eur_inr_prev` | Current published rate and the previous fixing for historical comparisons |
| **Markets** | EUR/USD, USD/INR, Brent oil, US yields, dollar index, ECB and RBI policy rates | Currency connections, global market conditions, and interest-rate differences |
| **FPI** | Weekly and four-week FPI flows; flows since prior fixing | Foreign portfolio investment activity in India |
| **Growth** | GDP, inflation, industrial production, unemployment and changes | Economic fundamentals and differences between India and the euro area |
| **Risk** | VIX and credit spread indicators | Financial-market stress and risk sentiment |
| **Trade** | Exports, imports, current-account and trade balances | External-sector conditions related to currency demand |
| **Reserves** | RBI reserves, components and changes | Foreign-exchange reserve levels and movements |
| **Uncertainty** | Geopolitical risk and economic policy uncertainty indices | Wider uncertainty in international markets |
| **Calendar** | Weekday, month, payday window, holidays, Diwali | Timing and seasonal context for remittances |
| **Events** | Days to scheduled decisions, latest policy changes, events since previous fixing | Known upcoming releases and recent economic news |

The final-cleaning script **does not remove entire economic categories**. It mainly removes dates without an EUR–INR target, redundant columns, and variables that do not vary across the retained fixing days. Later EDA and modeling will determine whether each remaining candidate feature is actually useful.

## 6.5 Load the final feature dataset

**What:** Read the CSV file produced by `daily_report_final_clean.py`.

**Why:** We want to inspect the saved output instead of repeating the full preprocessing script in the notebook. The file is expected in the project's `data/processed/` folder.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_PATH = PROCESSED_DIR / "features_ecb_clean.csv"

# Read the CSV output of the final cleaning script.
features = pd.read_csv(FEATURES_PATH, parse_dates=["date"])

print("Rows and columns:", features.shape)
print("First date:", features["date"].min().date())
print("Last date:", features["date"].max().date())
display(features.head())

## 6.6 Check the structure and missing values

**What:** Verify that the dates are unique, correctly ordered, and on weekdays, then inspect remaining missing values.

**Why:** Each row should represent one available ECB fixing. We should also understand whether missing values occur throughout the dataset or only at the beginning of a rolling feature.

In [ ]:
# Basic structure checks.
print("Duplicate dates:", features["date"].duplicated().sum())
print("Dates sorted:", features["date"].is_monotonic_increasing)
print("Weekend rows:", (features["date"].dt.dayofweek >= 5).sum())
print("Missing EUR–INR target values:", features["eur_inr"].isna().sum())

# Show only columns with missing observations.
missing = features.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
display(missing.to_frame("missing_count"))

### Observation from the supplied CSV

The uploaded version of `features_ecb_clean.csv` contains **1,746 rows and 95 columns**, dated **8 January 2020 to 29 October 2026**. There are **no duplicate dates, no weekend observations, and no missing EUR–INR target values**.

The only missing entries are **15 values each** in `fpi_equity_4w`, `fpi_debt_4w`, and `fpi_total_4w` (**45 missing cells in total**). They occur in the initial warm-up period, ending on **28 January 2020**, because a four-week measure requires enough prior observations. These values should not be interpreted as unexpected missing data.

## 6.7 Compare the merged data with the final feature file

**What:** Compare the number of rows and columns in the previous `data_daily.csv` and new `features_ecb_clean.csv`.

**Why:** This helps explain exactly how the final-cleaning step changed the structure of the dataset.

In [ ]:
DAILY_PATH = PROCESSED_DIR / "data_daily.csv"
data_daily = pd.read_csv(DAILY_PATH, parse_dates=["date"])

comparison = pd.DataFrame({
    "Dataset": ["data_daily.csv", "features_ecb_clean.csv"],
    "Rows": [len(data_daily), len(features)],
    "Columns": [data_daily.shape[1], features.shape[1]],
    "EUR–INR observations": [data_daily["eur_inr"].notna().sum(), features["eur_inr"].notna().sum()]
})
display(comparison)

# Identify which columns were removed during the last step.
removed_columns = sorted(set(data_daily.columns) - set(features.columns))
print("Columns removed:", len(removed_columns))
print(removed_columns)

### Interpretation

In the supplied files, the daily merged table has **2,500 rows and 106 columns**, while the final table has **1,746 rows and 95 columns**.

Of the 2,500 original calendar days, **750 had no published EUR–INR rate**. A further **four early fixing-day rows** (2, 3, 6, and 7 January 2020) were excluded by the chosen start date. The final dataset therefore contains 1,746 usable fixing-day observations. **Eleven columns** were also removed, mainly because they were replaced by `_since_fix` summaries or were constant on fixing days.

The smaller file is more appropriate for analysis of exchange-rate fixings, but it remains a **candidate-feature table** rather than a completed machine-learning dataset.

## 6.8 What does the final script validate?

The script includes `assert` statements that stop execution when key requirements fail. The main checks are:

1. **Completeness:** No missing values outside the three allowed four-week FPI warm-up columns; no remaining FPI warm-up gaps after 28 January 2020.
2. **Dates:** No duplicates, chronological order, no weekend rows, and no dates before 8 January 2020.
3. **Previous exchange rate:** `eur_inr_prev` matches the preceding actual fixing, not an arbitrary previous calendar date.
4. **Calendar fields:** Weekday, day-of-month, month and December flags match the date.
5. **Valid values:** Binary `is_` columns contain only 0 or 1; `days_to_` values are non-negative; policy-rate changes are multiples of five basis points; selected strictly positive indicators contain no zero or negative values.
6. **Event preservation:** Totals in the `_since_fix` event columns agree with the corresponding original daily events over the retained period.

These checks are useful safeguards, but **they do not independently prove that all source publication dates are accurate or that no feature leakage is possible**. That should be reviewed when constructing the prediction targets and applying time-based splits.

In [ ]:
# Repeat some important checks using the saved CSV.
assert features["date"].is_monotonic_increasing
assert not features["date"].duplicated().any()
assert (features["date"].dt.dayofweek < 5).all()
assert features["eur_inr"].notna().all()
assert (features["eur_inr"] > 0).all()

warmup_cols = ["fpi_equity_4w", "fpi_debt_4w", "fpi_total_4w"]
other_cols = [c for c in features.columns if c not in warmup_cols]
assert not features[other_cols].isna().any().any()
assert not features.loc[features["date"] > "2020-01-28", warmup_cols].isna().any().any()

print("Basic final-dataset checks passed.")

## 6.9 Gap log and summary

In addition to the cleaned CSV, `daily_report_final_clean.py` saves `docs/gap_log.csv`. This log records the original missing-value counts, the missing values on fixing days, whether the final file retains a column, and why some columns were removed. It provides a record of the transformation from the merged daily file to the final feature table.

In [ ]:
GAP_LOG_PATH = PROJECT_ROOT / "docs" / "gap_log.csv"

if GAP_LOG_PATH.exists():
    gap_log = pd.read_csv(GAP_LOG_PATH)
    display(gap_log.head(10))
    print("Logged columns:", len(gap_log))
else:
    print("Gap log not found. It is generated when daily_report_final_clean.py is run.")

### Summary of the final-cleaning stage

The script `daily_report_final_clean.py` converts the calendar-day merged data into a consistent ECB fixing-day dataset by keeping rows with a published EUR–INR rate, applying an initial warm-up cutoff, removing redundant or constant columns, organizing the retained variables into subject groups, and validating their dates and values.

The resulting `features_ecb_clean.csv` contains **1,746 observations and 95 columns**, with only the documented initial missing values for three four-week FPI variables. This file forms the **starting point for exploratory data analysis and subsequent feature engineering**.

---
# 7. Exploratory Data Analysis (EDA)

## 7.1 
